In [21]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [5]:
from src.data.cleaning import load_and_clean_data
from src.data.aggregation import aggregate_to_order_level
from src.data.splitting import time_based_split

from src.features.preprocessing import (
    add_time_features,
    learn_rare_countries,
    group_rare_countries,
    build_preprocessor,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
)

from src.models.evaluation import evaluate_classifier

In [6]:
import time


def train_and_evaluate_with_timing(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
):
    """Train, time, and evaluate a classification model."""

    fit_start = time.perf_counter()

    model.fit(
        X_train,
        y_train,
    )

    fit_time = time.perf_counter() - fit_start

    predict_start = time.perf_counter()

    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    prediction_time = time.perf_counter() - predict_start

    metrics = {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, y_proba),
        "fit_time_seconds": fit_time,
        "prediction_time_seconds": prediction_time,
    }

    return metrics

In [7]:
df_clean = load_and_clean_data()

order_df = aggregate_to_order_level(df_clean)

train_df, val_df, test_df = time_based_split(order_df)

In [8]:
train_df = add_time_features(train_df)
val_df = add_time_features(val_df)
test_df = add_time_features(test_df)

rare_countries = learn_rare_countries(train_df)

train_df = group_rare_countries(train_df, rare_countries)
val_df = group_rare_countries(val_df, rare_countries)
test_df = group_rare_countries(test_df, rare_countries)

In [9]:
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

X_train = train_df[FEATURE_COLUMNS]
X_val = val_df[FEATURE_COLUMNS]
X_test = test_df[FEATURE_COLUMNS]

y_train = train_df[TARGET]
y_val = val_df[TARGET]
y_test = test_df[TARGET]

preprocessor = build_preprocessor()

X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

In [10]:
print("X_train:", X_train_prepared.shape)
print("X_val:", X_val_prepared.shape)
print("X_test:", X_test_prepared.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (46026, 168)
X_val: (9863, 168)
X_test: (9863, 168)
y_train: (46026,)
y_val: (9863,)
y_test: (9863,)


## Baseline model

In [11]:
from src.models.baseline import (
    build_baseline_model,
)

baseline_model = build_baseline_model()
baseline_results = train_and_evaluate_with_timing(
    baseline_model,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val
)

baseline_results

{'accuracy': 0.7008009733346852,
 'precision': 0.8362930077691454,
 'recall': 0.5607441860465117,
 'f1': 0.6713442476890522,
 'roc_auc': 0.7429475604195166,
 'fit_time_seconds': 0.11917908396571875,
 'prediction_time_seconds': 0.0005828337743878365}

## 1- Decision Tree

In [13]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree_result = train_and_evaluate_with_timing(
    decision_tree,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val
)

decision_tree_result

{'accuracy': 0.6343911588766096,
 'precision': 0.6626816258966342,
 'recall': 0.6703255813953488,
 'f1': 0.6664816870144284,
 'roc_auc': 0.6308401525515068,
 'fit_time_seconds': 1.6667683338746428,
 'prediction_time_seconds': 0.0037467926740646362}

## 2-Random Forest

In [17]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=1
)

rensom_forest_result = train_and_evaluate_with_timing(
    random_forest,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val
)

rensom_forest_result

{'accuracy': 0.7038426442258947,
 'precision': 0.7904829545454546,
 'recall': 0.6212093023255814,
 'f1': 0.6956974684863007,
 'roc_auc': 0.7615809186253782,
 'fit_time_seconds': 86.05057420954108,
 'prediction_time_seconds': 1.0611689584329724}

## 3- XGBoost

In [16]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
)

xgb_results = train_and_evaluate_with_timing(
    xgb_model,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val,
)

xgb_results

{'accuracy': 0.7100273750380209,
 'precision': 0.8064830611747502,
 'recall': 0.6156279069767442,
 'f1': 0.6982485756488711,
 'roc_auc': 0.763946772789454,
 'fit_time_seconds': 0.24464499950408936,
 'prediction_time_seconds': 0.009898750111460686}

# Hyper Paramerer tuning

## 1- XGBoost hyperparameter tuning
n_estimators
max_depth
learning_rate 
min_child_weight 
subsample
colsample_bytree 

In [25]:
from sklearn.model_selection import ParameterGrid
from xgboost import XGBClassifier
import time


xgb_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8],
}


xgb_tuning_results = []

tuning_start = time.perf_counter()

for params in ParameterGrid(xgb_param_grid):

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
    )

    results = train_and_evaluate_with_timing(
        model,
        X_train_prepared,
        y_train,
        X_val_prepared,
        y_val,
    )

    xgb_tuning_results.append({
        **params,
        **results,
    })

total_xgb_tuning_time = time.perf_counter() - tuning_start

In [26]:
xgb_tuning_results_df = (
    pd.DataFrame(xgb_tuning_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print(f"Total tuning time: {total_xgb_tuning_time:.2f} seconds")
print(f"Total tuning time: {total_xgb_tuning_time / 60:.2f} minutes")

xgb_tuning_results_df.head(10)

Total tuning time: 8.31 seconds
Total tuning time: 0.14 minutes


,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample,accuracy,precision,recall,f1,roc_auc,fit_time_seconds,prediction_time_seconds
0,0.8,0.05,5,1,200,1.0,0.718443,0.845112,0.591814,0.696137,0.771479,0.356204,0.015420
1,0.8,0.05,5,5,100,1.0,0.718950,0.847345,0.590698,0.696119,0.771214,0.194150,0.008234
2,0.8,0.05,5,1,100,1.0,0.718950,0.847345,0.590698,0.696119,0.771164,0.204283,0.010720
3,0.8,0.05,3,1,200,0.8,0.719152,0.848542,0.589953,0.696005,0.770596,0.304497,0.009036
4,0.8,0.05,5,5,100,0.8,0.719152,0.848542,0.589953,0.696005,0.770449,0.206407,0.009188
5,0.8,0.05,3,1,200,1.0,0.719152,0.848542,0.589953,0.696005,0.770239,0.273302,0.008440
6,0.8,0.10,3,1,100,0.8,0.719051,0.848315,0.589953,0.695929,0.770184,0.134871,0.004655
7,0.8,0.05,5,5,200,1.0,0.718341,0.845070,0.591628,0.695995,0.770182,0.365577,0.012605
8,0.8,0.05,5,1,100,0.8,0.719152,0.848542,0.589953,0.696005,0.770041,0.213700,0.009010
9,0.8,0.10,5,1,100,1.0,0.718443,0.844197,0.592744,0.696470,0.769981,0.231616,0.007463


In [29]:
best_row = xgb_tuning_results_df.iloc[0]

best_xgb_params = {
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "min_child_weight": int(best_row["min_child_weight"]),
    "subsample": float(best_row["subsample"]),
    "colsample_bytree": float(best_row["colsample_bytree"]),
}

best_xgb_params

{'n_estimators': 200,
 'max_depth': 5,
 'learning_rate': 0.05,
 'min_child_weight': 1,
 'subsample': 1.0,
 'colsample_bytree': 0.8}

In [30]:
best_xgb = XGBClassifier(
    **best_xgb_params,
    random_state=42,
    eval_metric="logloss",
)

best_xgb_results = train_and_evaluate_with_timing(
    best_xgb,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val,
)

best_xgb_train_metrics = evaluate_classifier(
    best_xgb,
    X_train_prepared,
    y_train,
)

print("Best parameters:")
print(best_xgb_params)

print("\nTrain:")
print(best_xgb_train_metrics)

print("\nValidation:")
print(best_xgb_results)

Best parameters:
{'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 1, 'subsample': 1.0, 'colsample_bytree': 0.8}

Train:
{'accuracy': 0.720201625168383, 'precision': 0.8580114656320574, 'recall': 0.5869746068216931, 'f1': 0.6970737674068498, 'roc_auc': 0.8070688127681473, 'confusion_matrix': array([[18331,  2452],
       [10426, 14817]])}

Validation:
{'accuracy': 0.7184426645037006, 'precision': 0.8451115834218916, 'recall': 0.5918139534883721, 'f1': 0.6961374329795382, 'roc_auc': 0.7714790241678067, 'fit_time_seconds': 0.3612865824252367, 'prediction_time_seconds': 0.013821584172546864}


## 1- Random Forest hyperparameter tuning
max_depth
min_samples_split
min_samples_leaf
max_features
n_estimators

In [31]:
from sklearn.ensemble import RandomForestClassifier

rf_param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt"],
}


In [35]:
rf_tuning_results = []

tuning_start = time.perf_counter()

for params in ParameterGrid(rf_param_grid):

    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1
    )

    results = train_and_evaluate_with_timing(
        model,
        X_train_prepared,
        y_train,
        X_val_prepared,
        y_val
    )

    rf_tuning_results.append({
        **params,
        **results
    })

    total_rf_tuning_time = time.perf_counter() - tuning_start


In [40]:
print(rf_tuning_results[0])

{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200, 'accuracy': 0.7191523877116496, 'precision': 0.8485416109178485, 'recall': 0.589953488372093, 'f1': 0.6960052677787533, 'roc_auc': 0.766073581229532, 'fit_time_seconds': 0.8286986667662859, 'prediction_time_seconds': 0.06522770971059799}


In [42]:
rf_tuning_results_df = (
    pd.DataFrame(rf_tuning_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print(f"Total tuning time: {total_rf_tuning_time:.2f} seconds")
print(f"Total tuning time: {total_rf_tuning_time / 60:.2f} minutes")

rf_tuning_results_df.head(10)

Total tuning time: 61.14 seconds
Total tuning time: 1.02 minutes


,max_depth,max_features,min_samples_leaf,n_estimators,accuracy,precision,recall,f1,roc_auc,fit_time_seconds,prediction_time_seconds
0,20.0,sqrt,5,200,0.719152,0.848355,0.590140,0.696072,0.769695,1.486226,0.083747
1,20.0,sqrt,5,300,0.719152,0.848355,0.590140,0.696072,0.769566,2.176566,0.113680
2,20.0,sqrt,10,300,0.719152,0.848542,0.589953,0.696005,0.768818,1.885463,0.102305
3,20.0,sqrt,10,200,0.719152,0.848542,0.589953,0.696005,0.768030,1.337454,0.076525
4,NaN,sqrt,10,300,0.719254,0.848582,0.590140,0.696148,0.767641,2.931279,0.124956
5,NaN,sqrt,5,300,0.718848,0.846195,0.591628,0.696376,0.767607,4.556533,0.144117
6,NaN,sqrt,10,200,0.719254,0.848582,0.590140,0.696148,0.767564,1.906282,0.101237
7,NaN,sqrt,5,200,0.718747,0.845602,0.592000,0.696432,0.767500,2.746953,0.102696
8,20.0,sqrt,1,200,0.719254,0.848210,0.590512,0.696282,0.766558,2.481238,0.088079
9,10.0,sqrt,1,200,0.719152,0.848542,0.589953,0.696005,0.766074,0.828699,0.065228


In [44]:
best_rf_row = rf_tuning_results_df.iloc[0]

best_rf_params = {
    "n_estimators": int(best_rf_row["n_estimators"]),
    "max_depth": int(best_rf_row["max_depth"])
    if pd.notna(best_rf_row["max_depth"])
    else None,
    "min_samples_leaf": int(best_rf_row["min_samples_leaf"]),
    "max_features": best_rf_row["max_features"],
}

best_rf_params

{'n_estimators': 200,
 'max_depth': 20,
 'min_samples_leaf': 5,
 'max_features': 'sqrt'}

In [45]:
best_rf = RandomForestClassifier(
    **best_rf_params,
    random_state=42,
    n_jobs=-1,
)

best_rf_results = train_and_evaluate_with_timing(
    best_rf,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val,
)

best_rf_train_metrics = evaluate_classifier(
    best_rf,
    X_train_prepared,
    y_train,
)

print("Best parameters:")
print(best_rf_params)

print("\nTrain:")
print(best_rf_train_metrics)

print("\nValidation:")
print(best_rf_results)

Best parameters:
{'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 5, 'max_features': 'sqrt'}

Train:
{'accuracy': 0.7182896623647503, 'precision': 0.8582015521970007, 'recall': 0.5826169631184883, 'f1': 0.6940537989617744, 'roc_auc': 0.8558671256073256, 'confusion_matrix': array([[18353,  2430],
       [10536, 14707]])}

Validation:
{'accuracy': 0.7191523877116496, 'precision': 0.8483551751805296, 'recall': 0.590139534883721, 'f1': 0.696071977177968, 'roc_auc': 0.769694565352568, 'fit_time_seconds': 1.7072179578244686, 'prediction_time_seconds': 0.0777542507275939}


## Decision Tree tuning

In [48]:
from sklearn.tree import DecisionTreeClassifier

dt_param_grid = {
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": [None, "sqrt"],
}

In [49]:
dt_tuning_results = []

tuning_start = time.perf_counter()

for params in ParameterGrid(dt_param_grid):

    model = DecisionTreeClassifier(
        **params,
        random_state=42,
    )

    results = train_and_evaluate_with_timing(
        model,
        X_train_prepared,
        y_train,
        X_val_prepared,
        y_val,
    )

    dt_tuning_results.append({
        **params,
        **results,
    })

total_dt_tuning_time = time.perf_counter() - tuning_start

In [50]:
dt_tuning_results_df = (
    pd.DataFrame(dt_tuning_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print(f"Total tuning time: {total_dt_tuning_time:.2f} seconds")
print(f"Total tuning time: {total_dt_tuning_time / 60:.2f} minutes")

dt_tuning_results_df.head(10)

Total tuning time: 18.04 seconds
Total tuning time: 0.30 minutes


,max_depth,max_features,min_samples_leaf,min_samples_split,accuracy,precision,recall,f1,roc_auc,fit_time_seconds,prediction_time_seconds
0,5.0,NaN,5,2,0.718037,0.845682,0.590326,0.695300,0.767310,0.069672,0.001065
1,5.0,NaN,5,10,0.718037,0.845682,0.590326,0.695300,0.767310,0.068710,0.001056
2,5.0,NaN,5,20,0.718037,0.845682,0.590326,0.695300,0.767310,0.069408,0.001262
3,5.0,NaN,1,2,0.718138,0.846092,0.590140,0.695309,0.766651,0.134102,0.001349
4,5.0,NaN,1,10,0.718138,0.846092,0.590140,0.695309,0.766651,0.070260,0.001729
5,5.0,NaN,1,20,0.718138,0.846092,0.590140,0.695309,0.766651,0.071965,0.001219
6,5.0,NaN,10,2,0.717125,0.843293,0.590698,0.694748,0.766023,0.069327,0.001109
7,5.0,NaN,10,10,0.717125,0.843293,0.590698,0.694748,0.766023,0.072785,0.001400
8,5.0,NaN,10,20,0.717125,0.843293,0.590698,0.694748,0.766023,0.072605,0.001106
9,10.0,NaN,5,20,0.713373,0.837037,0.588651,0.691207,0.763619,0.162463,0.001242


In [51]:
best_dt_row = dt_tuning_results_df.iloc[0]

best_dt_params = {
    "max_depth": int(best_dt_row["max_depth"])
    if pd.notna(best_dt_row["max_depth"])
    else None,
    "max_features": best_dt_row["max_features"]
    if pd.notna(best_dt_row["max_features"])
    else None,
    "min_samples_leaf": int(best_dt_row["min_samples_leaf"]),
    "min_samples_split": int(best_dt_row["min_samples_split"]),
}

best_dt_params

{'max_depth': 5,
 'max_features': None,
 'min_samples_leaf': 5,
 'min_samples_split': 2}

In [52]:
best_dt = DecisionTreeClassifier(
    **best_dt_params,
    random_state=42,
)

best_dt_results = train_and_evaluate_with_timing(
    best_dt,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val,
)

best_dt_train_metrics = evaluate_classifier(
    best_dt,
    X_train_prepared,
    y_train,
)

print("Best parameters:")
print(best_dt_params)

print("\nTrain:")
print(best_dt_train_metrics)

print("\nValidation:")
print(best_dt_results)

Best parameters:
{'max_depth': 5, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 2}

Train:
{'accuracy': 0.7185503845652458, 'precision': 0.8570515427973735, 'recall': 0.5842807907142574, 'f1': 0.6948553660604918, 'roc_auc': 0.7755044563055541, 'confusion_matrix': array([[18323,  2460],
       [10494, 14749]])}

Validation:
{'accuracy': 0.7180371083848728, 'precision': 0.845682302771855, 'recall': 0.5903255813953489, 'f1': 0.6952996603484168, 'roc_auc': 0.7673099531567384, 'fit_time_seconds': 0.11469700001180172, 'prediction_time_seconds': 0.0018645403906702995}


## Stage 2 tuning : selected model -> XGBoost 

In [59]:

xgb_stage2_param_grid = {
    "n_estimators": [150, 200, 250],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.07],
    "min_child_weight": [1, 3],
    "subsample": [1.0],
    "colsample_bytree": [0.7, 0.8, 0.9],
}


xgb_stage2_tuning_results = []

xgb_stage2_tuning_start = time.perf_counter()

for params in ParameterGrid(xgb_stage2_param_grid):

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
    )

    results = train_and_evaluate_with_timing(
        model,
        X_train_prepared,
        y_train,
        X_val_prepared,
        y_val,
    )

    xgb_stage2_tuning_results.append({
        **params,
        **results,
    })

total_xgb_stage2_tuning_time = (
    time.perf_counter() - xgb_stage2_tuning_start
)

In [60]:
xgb_stage2_results_df = (
    pd.DataFrame(xgb_stage2_tuning_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print(
    f"Total Stage 2 tuning time: "
    f"{total_xgb_stage2_tuning_time:.2f} seconds"
)

print(
    f"Total Stage 2 tuning time: "
    f"{total_xgb_stage2_tuning_time / 60:.2f} minutes"
)

xgb_stage2_results_df.head(10)

Total Stage 2 tuning time: 58.59 seconds
Total Stage 2 tuning time: 0.98 minutes


,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample,accuracy,precision,recall,f1,roc_auc,fit_time_seconds,prediction_time_seconds
0,0.7,0.07,5,1,150,1.0,0.718037,0.844214,0.591814,0.695833,0.771684,0.255179,0.010138
1,0.9,0.03,5,1,200,1.0,0.718950,0.847345,0.590698,0.696119,0.771671,0.339067,0.012663
2,0.7,0.05,4,1,200,1.0,0.718950,0.847902,0.590140,0.695919,0.771515,0.279255,0.010037
3,0.8,0.05,5,1,200,1.0,0.718443,0.845112,0.591814,0.696137,0.771479,0.325070,0.013245
4,0.7,0.07,5,1,200,1.0,0.717226,0.842607,0.591628,0.695158,0.771423,0.307240,0.015872
5,0.8,0.03,5,1,150,1.0,0.718645,0.847408,0.589953,0.695624,0.771398,0.279321,0.013539
6,0.8,0.05,5,1,150,1.0,0.718848,0.846564,0.591256,0.696243,0.771353,0.259840,0.010069
7,0.7,0.05,4,1,250,1.0,0.718544,0.845888,0.591256,0.696014,0.771312,0.339534,0.011451
8,0.7,0.07,4,1,150,1.0,0.718443,0.846585,0.590326,0.695605,0.771268,0.211205,0.007755
9,0.8,0.03,5,1,200,1.0,0.718443,0.846955,0.589953,0.695471,0.771221,0.382643,0.014100


In [61]:
best_xgb_stage2_row = xgb_stage2_results_df.iloc[0]

best_xgb_stage2_params = {
    "n_estimators": int(
        best_xgb_stage2_row["n_estimators"]
    ),
    "max_depth": int(
        best_xgb_stage2_row["max_depth"]
    ),
    "learning_rate": float(
        best_xgb_stage2_row["learning_rate"]
    ),
    "min_child_weight": int(
        best_xgb_stage2_row["min_child_weight"]
    ),
    "subsample": float(
        best_xgb_stage2_row["subsample"]
    ),
    "colsample_bytree": float(
        best_xgb_stage2_row["colsample_bytree"]
    ),
}

best_xgb_stage2_params

{'n_estimators': 150,
 'max_depth': 5,
 'learning_rate': 0.07,
 'min_child_weight': 1,
 'subsample': 1.0,
 'colsample_bytree': 0.7}

In [62]:
best_xgb_stage2 = XGBClassifier(
    **best_xgb_stage2_params,
    random_state=42,
    eval_metric="logloss",
)

best_xgb_stage2_results = train_and_evaluate_with_timing(
    best_xgb_stage2,
    X_train_prepared,
    y_train,
    X_val_prepared,
    y_val,
)

best_xgb_stage2_train_metrics = evaluate_classifier(
    best_xgb_stage2,
    X_train_prepared,
    y_train,
)

print("Best Stage 2 parameters:")
print(best_xgb_stage2_params)

print("\nTrain:")
print(best_xgb_stage2_train_metrics)

print("\nValidation:")
print(best_xgb_stage2_results)

Best Stage 2 parameters:
{'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.07, 'min_child_weight': 1, 'subsample': 1.0, 'colsample_bytree': 0.7}

Train:
{'accuracy': 0.720505801068961, 'precision': 0.8578366190668902, 'recall': 0.5878065206195777, 'f1': 0.6976022566995769, 'roc_auc': 0.8076409516217946, 'confusion_matrix': array([[18324,  2459],
       [10405, 14838]])}

Validation:
{'accuracy': 0.7180371083848728, 'precision': 0.8442144373673036, 'recall': 0.5918139534883721, 'f1': 0.6958328776112873, 'roc_auc': 0.7716837872569746, 'fit_time_seconds': 0.2911491673439741, 'prediction_time_seconds': 0.011345875449478626}
